In [1]:
from template_build import TemplateBuild
import os

def add_page(titre, href):
    return {'titre': titre, "href": href}

def make_menuitem(m, active=False):
    if active:
        classe = 'class="active"'
    else:
        classe = ''
    return f"""<li {classe}><a href="{m['href']}">{m['titre']}</a></li>"""
def make_menuderoulantitem(m, active=False):
    if active:
        classe = 'class="active"'
    else:
        classe = ''
    return f"""<li {classe}><a href="#" class="submenu-toggle">{m['titre']}<span class="arrow">></span></a><ul>"""

template_menu = """<a href="index.html">Accueil</a>"""

In [26]:
titre_site = "Martouzet Physique"
footer = "@ G. Martouzet 2026"
menu = [
    {
        "titre" : "Accueil",
        "href" : "index.html" 
    },
    {
        "titre" : "Physique PCSI/PSI",
        "contents" : [
            add_page( "Cours PCSI", "cours.html" ),
            add_page( "Fiches Méthodes", "fiches_methodes_PCSI.html"),
            add_page( "Méthode Euler", "methode_euler.html"),
        ]
    },
    {
        "titre" : "Mes publications",
        "contents" : [
            add_page( "Thèse", "thèse.html"),
            add_page( "Articles pédagogiques", "article_pédagogique.html"),
        ]
    },
    {
        "titre" : "Outils",
        "contents" : [
            add_page( "Figures tikz", "figure_tikz.html"),
            add_page( "Applications et codes", "code_python.html"),
            add_page( "Sites web extérieurs", "site_web.html"),
        ]
    }
]

page_a_compiler = {} # Key : la page; Value : la page d'origine à illuminer dans le menu
for m in menu:
    if 'href' in m:
        page_a_compiler[m['href']] = m['href']
    else:
        for i in m['contents']:
            page_a_compiler[i['href']] = i['href']
page_a_compiler['application_info.html'] = "code_python.html"

In [27]:
def build_menu( titre_site, menu, footer, page ):
    menu_HTML = ['<nav id="sidebar">', '\t<div>',f'\t<div id="title_site">{titre_site}</div>','\t<ul>']
    for m in menu:
        if 'href' in m:
            menu_HTML.append( '\t'+ make_menuitem(m, active=(page==m['href'])) )
        else:
            sub_menu = []
            menu_actif = False
            for i in m['contents']:
                page_active = (page_a_compiler[page]==i['href'])
                if page_active:
                    menu_actif = True
                sub_menu.append( '\t\t'+make_menuitem( i, active=page_active ) )
                
            menu_HTML.append( '\t'+ make_menuderoulantitem(m, active=menu_actif) )
            menu_HTML += sub_menu
            menu_HTML += ["\t</li>","</ul>"]

    menu_HTML.append( f"""
    </ul>
    <div class="footer">
                {footer}
            </div>

        </div>
    </nav>""")
    return {'MENU' : ''.join(menu_HTML)}

In [30]:
for page in page_a_compiler:
    print( page )
    if not os.path.isfile('./draft/'+page):
        print( '  ', 'Introuvable' )
        continue
    menu_compile = build_menu( titre_site, menu, footer, page )
    template = TemplateBuild( './draft/'+page )
    template.save( './docs/'+page, menu_compile )

index.html
cours.html
fiches_methodes_PCSI.html
methode_euler.html
thèse.html
article_pédagogique.html
figure_tikz.html
   Introuvable
code_python.html
site_web.html
application_info.html


In [77]:
import markdown

md_text = """
# some Python code
hi = 'Hello'
print(hi)
 - df
 - dsf
"""
html = markdown.markdown(md_text)
print(html)

<h1>some Python code</h1>
<p>hi = 'Hello'
print(hi)
 - df
 - dsf</p>


In [30]:
page_a_compiler

{'index.html': 'index.html',
 'cours.html': 'cours.html',
 'fiches_methodes_PCSI.html': 'fiches_methodes_PCSI.html',
 'methode_euler.html': 'methode_euler.html',
 'thèse.html': 'thèse.html',
 'article_pédagogique.html': 'article_pédagogique.html',
 'figure_tikz.html': 'figure_tikz.html',
 'code_python.html': 'code_python.html',
 'application_info.html': 'code_python.html'}

# Compilation code et application

In [11]:
import csv

liste_file = []
liste_file_maj = []
with open('docs/codes_python/data_code.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile, delimiter=';')
    for row in reader:
        liste_file.append( row )
for row in liste_file:
    id = abs(hash(str(row)) )
    code = {}
    for k, v in row.items():
        if v != '':
            code[k] = v
    code['id'] = id
    code['Keywords'] = code['Keywords'].split(',')
    
    liste_file_maj.append( code )


with open( 'docs/codes_python/liste_application.js', 'w', encoding='UTF8') as f:
    f.write( 'const data_application = [\n')
    #f.write( str(liste_file) )
    for row in liste_file_maj:
        f.write( str(row)+ ',\n' )
    f.write( ']' )
